# 02 · Feature engineering experiments

**Question:** Does adding information inferred from passenger names, family size, cabin or fare improve Gradient Boosting?

Keep the classifier and CV protocol fixed, then compare the original seven features against one feature group at a time and against all engineered features combined. Feature construction and preprocessing both happen inside the CV pipeline.

**Original experiment:** `06_featuretesting.ipynb`.

## 1. Import helpers and load data

In [ ]:
import sys
import pandas as pd
from IPython.display import display
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score

In [ ]:
from pathlib import Path

# Works when running Jupyter from either the repository root or notebooks/.
cwd = Path.cwd().resolve()
candidates = [cwd, *cwd.parents]
project_dir = next(
    (folder for folder in candidates if (folder / 'data' / 'train.csv').exists()),
    None,
)
if project_dir is None:
    raise FileNotFoundError(
        'data/train.csv not found. Download train.csv and test.csv from '
        'https://www.kaggle.com/competitions/titanic/data and put them in data/.'
    )

In [ ]:
sys.path.insert(0, str(project_dir / 'src'))
from feature_engineering import FEATURE_SETS, create_experiment_pipeline

train = pd.read_csv(project_dir / 'data' / 'train.csv')
X = train.drop(columns='Survived')
y = train['Survived']

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
experiments = [
    'Baseline', 'Family', 'Women and children',
    'Class and fare', 'Title', 'Cabin', 'Combined',
]

## 2. Feature groups

- **Family:** family size, travelling alone, parent/child aboard.
- **Women and children:** age/sex and parent-child indicators.
- **Class and fare:** sex-by-class interaction and fare per family member.
- **Title:** honorific extracted from passenger name.
- **Cabin:** deck and missing-cabin indicator.
- **Combined:** all engineered groups together.

`src/feature_engineering.py` contains the original feature definitions and leakage-safe modelling pipeline.

In [ ]:
display(pd.DataFrame([
    {'Feature group': name, 'Extra features': ', '.join(FEATURE_SETS[name]) or 'None'}
    for name in experiments
]))

## 3. Run the same five-fold CV for every feature group

In [ ]:
results = []
for name in experiments:
    pipeline = create_experiment_pipeline(
        feature_set=name,
        classifier=GradientBoostingClassifier(random_state=42),
    )
    scores = cross_val_score(pipeline, X, y, cv=cv, scoring='accuracy')
    results.append({
        'Feature group': name,
        'Mean CV accuracy': scores.mean(),
        'Fold std': scores.std(),
        'Fold scores': scores.round(4).tolist(),
    })

results_df = pd.DataFrame(results)
baseline = results_df.loc[
    results_df['Feature group'].eq('Baseline'), 'Mean CV accuracy'
].iloc[0]
results_df['Change vs baseline (pp)'] = (
    results_df['Mean CV accuracy'] - baseline
) * 100
results_df = results_df.sort_values('Mean CV accuracy', ascending=False)
display(results_df.round(4))

### Outcome

**Recorded original five-fold CV:** Title **0.8361**, Family **0.8339**, Baseline **0.8272**, and Combined **0.8283**. Title initially appeared useful; adding everything did not yield the strongest result.

These are exploratory results on one five-fold split. Testing multiple alternatives against the same CV split can produce optimistic feature choices. The next notebook repeats the comparison across five different CV repetitions.